# GeoVision — Train ResNet50 (13-channel) on EuroSAT MS

Runs on Google Colab T4 GPU. ~30–45 min for 20 epochs.

Steps:
1. Mount Drive
2. Clone GeoVision into Drive (or upload manually before GitHub repo exists)
3. Install deps
4. Download EuroSAT MS (cached on Drive after first run)
5. Train
6. Evaluate

All outputs persist at `/content/drive/MyDrive/GeoVision/code/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/GeoVision'
os.makedirs(f'{DRIVE_ROOT}/data/raw', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Set REPO_URL after creating the GitHub repo. Until then, manually upload the
# GeoVision project folder to /content/drive/MyDrive/GeoVision/code/.
REPO_URL = 'https://github.com/Caio99-git/geovision.git'
REPO_DIR = f'{DRIVE_ROOT}/code'

if REPO_URL and not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
elif REPO_URL:
    !git -C {REPO_DIR} pull

assert os.path.exists(REPO_DIR), 'Set REPO_URL above, or upload code/ to Drive first'
%cd {REPO_DIR}
import sys
sys.path.insert(0, REPO_DIR)

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.93 KiB | 11.00 KiB/s, done.
From https://github.com/Caio99-git/geovision
   5153a9c..a59fbfb  main       -> origin/main
Updating 5153a9c..a59fbfb
Fast-forward
 notebooks/02_train_pytorch_colab.ipynb    | 33 ++++++++++---------------------
 notebooks/03_train_tensorflow_colab.ipynb | 33 ++++++++++---------------------
 2 files changed, 20 insertions(+), 46 deletions(-)
/content/drive/MyDrive/GeoVision/code


In [3]:
# Colab preinstalls torch, tensorflow, numpy, pandas, sklearn, opencv, matplotlib, pillow
%pip install -q 'rasterio>=1.4.0' 'albumentations>=1.4.0' 'mlflow>=2.18.0'

In [4]:
import glob

EUROSAT_URL = 'https://madm.dfki.de/files/sentinel/EuroSATallBands.zip'
ZIP_PATH = f'{DRIVE_ROOT}/data/raw/EuroSATallBands.zip'
EXTRACT_DIR = f'{DRIVE_ROOT}/data/raw'

candidates = glob.glob(f'{EXTRACT_DIR}/**/Forest', recursive=True)
if not candidates:
    if not os.path.exists(ZIP_PATH):
        print('Downloading EuroSAT MS (~3 GB)...')
        !wget -q -O {ZIP_PATH} {EUROSAT_URL}
    print('Extracting...')
    !unzip -q {ZIP_PATH} -d {EXTRACT_DIR}
    candidates = glob.glob(f'{EXTRACT_DIR}/**/Forest', recursive=True)

DATA_ROOT = os.path.dirname(candidates[0])
print('Data root:', DATA_ROOT)
!ls {DATA_ROOT}

Extracting...
[/content/drive/MyDrive/GeoVision/data/raw/EuroSATallBands.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/drive/MyDrive/GeoVision/data/raw/EuroSATallBands.zip or
        /content/drive/MyDrive/GeoVision/data/raw/EuroSATallBands.zip.zip, and cannot find /content/drive/MyDrive/GeoVision/data/raw/EuroSATallBands.zip.ZIP, period.


IndexError: list index out of range

In [ ]:
!python -m src.train --framework pytorch --data-root {DATA_ROOT} --epochs 20 --batch-size 64 --lr 1e-4

In [ ]:
!python -m src.evaluate --framework pytorch --data-root {DATA_ROOT} --ckpt models/resnet50_13ch.pt

## Done

- **Weights:** `code/models/resnet50_13ch.pt` (~100 MB)
- **Metrics:** `code/data/metrics/confusion_matrix_pytorch.{csv,png}`, `summary_pytorch.csv`, `classification_report_pytorch.csv`
- **MLflow runs:** `code/mlruns/`

All persisted on Drive. Sync `data/metrics/*.csv` and `*.png` back to your local clone and commit (weights stay in `.gitignore` — distribute via Hugging Face Hub in Phase 6).